# DPO — Direct Preference Optimization

> **SFT teaches the model *how* to talk. Alignment teaches it *what* is valued, safe, and helpful.**
> DPO does alignment **without** a reward model, a value model, or an RL loop.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)
- **DPO (Direct Preference Optimization)** re-parameterizes the **RLHF objective** so that the **reward model and the policy are the same network**. It replaces PPO's actor–critic reinforcement-learning loop with a **single supervised binary-classification loss**.
- The engine is a **maximum-likelihood classifier** over **pairwise preferences** `(x, y_w, y_l)` — prompt, **w**inning (chosen) completion, **l**osing (rejected) completion — trained with **binary cross-entropy** on an **implicit reward** defined by the model's own log-probabilities.
- The **implicit reward** of a completion is the **log-ratio of the policy to a frozen reference policy**, scaled by a temperature **β**:
  $$\hat r_\theta(x,y) \;=\; \beta \,\log \frac{\pi_\theta(y\mid x)}{\pi_{\text{ref}}(y\mid x)}$$
- No reward network is ever instantiated — the reward is *read off* the two log-prob streams. Hence **"reward-free" / "direct."**

### One-sentence definition of the mechanics
> **DPO minimizes the binary cross-entropy of a Bradley–Terry preference classifier whose logits are the β-scaled difference of policy-vs-reference log-probability ratios between the chosen and rejected completions.**

The loss:
$$\mathcal{L}_{\text{DPO}} = -\,\mathbb{E}_{(x,y_w,y_l)}\Big[\log \sigma\big(\beta\,\log\tfrac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta\,\log\tfrac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\big)\Big]$$

### The exact engineering problem it solves
Classic **RLHF-PPO** requires, *in memory at once*:
- **Policy (actor)**, **frozen reference** (for the KL penalty), **reward model**, and a **value/critic head** — **up to 4 models**.
- **Online generation** (rollouts) every step, plus PPO's advantage estimation, clipping, and reward normalization — **notoriously unstable** and hyper-parameter-fragile.

DPO removes all of it:
- **No reward-model training stage**, **no value model**, **no sampling/rollouts**, **no RL**.
- Training becomes **offline and deterministic** over a static `(chosen, rejected)` dataset — a stable classification problem you can debug like any supervised run.

---

### The Human Element — industry-standard Hugging Face preference datasets

| HF path | What it is | Why it's shaped this way for DPO |
|---|---|---|
| **`HuggingFaceH4/ultrafeedback_binarized`** | The canonical DPO corpus (the **Zephyr** recipe). GPT-4 scored 4 model completions per prompt on helpfulness/honesty; the **highest → `chosen`**, a **random lower → `rejected`**. | DPO needs exactly **one preferred + one dispreferred** completion per prompt. "**binarized**" means it has already been reduced from ranked lists to the **pairwise** `(chosen, rejected)` form DPO consumes. Split **`train_prefs`** is the preference split (`test_prefs` for eval). |
| **`Anthropic/hh-rlhf`** | **Real human** helpfulness & harmlessness pairs — crowdworkers picked the better of two assistant replies. | Genuinely **human** preference labels (not AI-scored), so the model aligns to *human* judgment. Stored as `chosen`/`rejected` raw transcript strings sharing the same prompt prefix — the pairwise structure DPO requires. |
| **`argilla/dpo-mix-7k`** | A small (~7k), **curated, cleaned** mix (UltraFeedback + Capybara style) explicitly packaged for DPO. | Deliberately tiny and high-signal — **ideal for a T4 quick run**: same `(prompt, chosen, rejected)` schema, far fewer noisy pairs, so a single epoch converges fast. |

**Why the pairwise schema is non-negotiable:** DPO's loss is a *contrast*. The gradient only exists because two completions **share a prompt** and differ in preference — the classifier learns the **margin** `r̂(y_w) − r̂(y_l)`. A dataset of single "good" answers (SFT-style) carries **no preference signal** and cannot train DPO.

> This notebook trains on **`HuggingFaceH4/ultrafeedback_binarized` / `train_prefs`** (Section 3).

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the mathematical reason DPO works
- Start from the **KL-constrained reward-maximization** objective that PPO also optimizes:
  $$\max_{\pi_\theta}\; \mathbb{E}_{x,\,y\sim\pi_\theta}\big[r(x,y)\big] \;-\; \beta\,\mathbb{D}_{\text{KL}}\!\big[\pi_\theta(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\big]$$
- This has a **known closed-form optimum**: $\pi^*(y|x)=\tfrac{1}{Z(x)}\,\pi_{\text{ref}}(y|x)\,\exp\!\big(\tfrac{1}{\beta}r(x,y)\big)$.
- **Invert it** to express the reward in terms of the policy: $r(x,y)=\beta\log\tfrac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)}+\beta\log Z(x)$.
- Plug that into the **Bradley–Terry** preference model $P(y_w\!\succ\!y_l)=\sigma\big(r(x,y_w)-r(x,y_l)\big)$. The **intractable partition function `Z(x)` cancels** (it depends only on `x`, shared by both completions).
- What remains is the DPO loss — **a classification objective over log-prob ratios**. The reward model was never needed; it was **implicit in the policy the whole time**.

#### VRAM & Compute Impact (vs RLHF-PPO baseline)
- **Model count in memory:** PPO holds **~4** (policy + reference + reward + value). DPO holds **1 trainable policy + 1 reference**. With **PEFT/LoRA**, the reference is the *same base weights with adapters disabled* → **effectively ONE model in VRAM** (the trick that makes this fit a T4).
- **No rollouts:** PPO generates samples every step (the dominant cost, and the source of variance). DPO reads **precomputed static pairs** — **no generation at train time**.
- **Forward passes:** DPO runs **2 forwards per step** (policy + reference) over the **chosen ⧺ rejected** concatenation → roughly **2× activation memory of plain SFT**, but still **far below PPO's 4-model + KV-cache-for-generation footprint**.
- **Optimizer state:** with **QLoRA (4-bit base) + `paged_adamw_8bit`**, only the small LoRA adapters carry optimizer state — kilobytes-to-megabytes, not gigabytes.
- **Net:** on a **16 GB T4**, a 0.5B–1.5B model does DPO comfortably; the equivalent PPO run is tight-to-impossible without aggressive offloading.

#### Pros & Cons (production trade-offs)

**Pros**
- **Single stage, single model** — no separate reward-model training, no RL infra.
- **Stable BCE loss** — converges like supervised learning; easy to monitor and reproduce.
- **No reward hacking** of an external RM; nothing to over-optimize against.
- **Cheap & fast** — offline, no sampling loop; runs on commodity GPUs.

**Cons**
- **Off-policy / no exploration** — learns only from the **fixed** pairs; it never discovers behaviors outside the dataset (PPO can).
- **Reference-quality dependent** — a weak/mismatched `π_ref` (SFT start) caps final quality; **distribution shift** between the SFT model and the preference data hurts.
- **β is sensitive** — too high freezes the policy near `π_ref`; too low collapses/over-fits it.
- **Likelihood-displacement pathology** — vanilla DPO can *lower* the probability of chosen responses while widening the margin; **verbosity/length bias** is common. Mitigations: **`loss_type` variants** — **IPO** (regularizes the margin), **cDPO** (label smoothing for noisy prefs), **robust**, **KTO** (unpaired) — all exposed via `DPOConfig(loss_type=...)`.

#### Metrics DPO emits (watch these during training)
- **`rewards/chosen`, `rewards/rejected`** — the implicit `β·log-ratio` for each side.
- **`rewards/margins`** = chosen − rejected → **should climb** (the model separates good from bad).
- **`rewards/accuracies`** = fraction of pairs with margin > 0 → **should trend toward 1.0**.
- **`logps/chosen`, `logps/rejected`** — raw log-probs; watch that `logps/chosen` doesn't crater (displacement).

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit) + TRL `DPOTrainer` + the adapter-disabling reference trick.**

> ⚙️ **Reference model for free:** `ref_model=None` on a **PEFT** policy makes TRL compute the KL term by **temporarily disabling the LoRA adapters** — the frozen 4-bit base weights *are* `π_ref`. One model in VRAM, two behaviors.

**Executable pipeline:**

| Step | What | Maps to |
|---|---|---|
| 1 | 4-bit `Qwen2.5-0.5B-Instruct` + LoRA + tokenizer | policy **and** (adapters-off) reference |
| 2 | `HuggingFaceH4/ultrafeedback_binarized` → `(prompt, chosen, rejected)` | the preference signal |
| 3 | `DPOConfig` (β, `loss_type`, T4 memory flags) | the objective |
| 4 | `DPOTrainer.train()` | the single classification loss |
| 5 | Save adapter · export · inference | ship it |

### Environment Setup

In [ ]:
# Modern, up-to-date stack. On Colab T4, torch+CUDA are preinstalled; we only add the RLHF libs.
%pip install transformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

set_seed(42)
# Reduce CUDA fragmentation OOMs on the T4 (DPO holds 2x activations: chosen + rejected).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

### Step 1 — Quantization, LoRA & Tokenizer

We start from the **Instruct** checkpoint — SFT is *already done*, so it is both a coherent **policy** to refine and a coherent **reference** `π_ref`. One 4-bit base + one LoRA config serves both roles (adapters on = policy, adapters off = reference).

In [ ]:
# The SFT/Instruct start point. This frozen base = the DPO reference policy (pi_ref).
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 4-bit NF4: the base weights (shared by policy AND reference) live in 4-bit on the T4.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",  # 4-bit NormalFloat (QLoRA)
    bnb_4bit_use_double_quant=True,  # nested quantization -> extra ~0.4 GB saved
    bnb_4bit_compute_dtype=torch.bfloat16,  # matmuls upcast to bf16 (T4 supports bf16)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # DPO pads batched chosen/rejected; Qwen needs a pad id

# LoRA = the ONLY trainable tensors. Disabling these adapters reproduces pi_ref exactly,
# which is why we never load a second model copy for the KL term.
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],  # attention + MLP projections
)

# Load the base ONCE. TRL wraps it with LoRA (policy) and toggles adapters off for the reference.
policy_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # PyTorch SDPA: mem-efficient attention kernels (T4 has no FlashAttn-2)
)
policy_model.config.use_cache = False  # required once gradient checkpointing is on
print(policy_model.get_memory_footprint() / 1e9, "GB base (4-bit)")

### Step 2 — Dataset: `HuggingFaceH4/ultrafeedback_binarized`

The **`train_prefs`** split holds one **preferred** and one **dispreferred** completion per prompt. Its `chosen`/`rejected` fields are **conversational** (lists of `{role, content}`) that share the same user turn. We flatten each into DPO's **standard** triple:

- **`prompt`** — the user turn wrapped in Qwen's chat template (`add_generation_prompt=True`).
- **`chosen`** / **`rejected`** — the **raw text** of the two final assistant turns.

In [ ]:
# train_prefs = the preference split (test_prefs exists for eval).
raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
raw = raw.shuffle(seed=42).select(range(800))  # subset so a T4 finishes in a few minutes

def to_dpo_triple(ex):
    # ultrafeedback_binarized: ex["chosen"] = [{user}, {assistant}] (single-turn here).
    # Everything but the final assistant message is the SHARED prompt.
    prompt_msgs = ex["chosen"][:-1]
    return {
        # Same chat template the model was instruct-tuned on; ends with the assistant header
        # the model must complete -> policy & reference see identical prompt formatting.
        "prompt":   tokenizer.apply_chat_template(prompt_msgs, tokenize=False,
                                                  add_generation_prompt=True),
        "chosen":   ex["chosen"][-1]["content"],    # preferred completion (raw text)
        "rejected": ex["rejected"][-1]["content"],  # dispreferred completion (raw text)
    }

# Passing string prompt/chosen/rejected => TRL treats it as the STANDARD (non-conversational)
# format and does NOT re-apply a chat template, so no double-templating.
dpo_ds = raw.map(to_dpo_triple, remove_columns=raw.column_names)
print(dpo_ds)
print("\n--- prompt[0] ---\n", dpo_ds[0]["prompt"][:400])
print("\n--- chosen[0] ---\n", dpo_ds[0]["chosen"][:200])

### Step 3 — `DPOConfig` (the objective) & `DPOTrainer`

**β** is the KL temperature; **`loss_type="sigmoid"`** is vanilla DPO (Bradley–Terry BCE). Because both **chosen and rejected** are tokenized every step, sequence caps and micro-batching are the levers that keep the T4 in budget.

In [ ]:
dpo_config = DPOConfig(
    output_dir="./dpo_output",
    run_name="dpo-t4",

    # ---- The DPO knobs ----
    beta=0.1,  # KL strength: higher -> hug pi_ref; lower -> trust the pairs more
    loss_type="sigmoid",  # vanilla DPO; swap to "ipo"/"robust"/"cdpo" to fight overfitting/noise

    # ---- Sequence budget (chosen AND rejected tokenized => ~2x activation memory) ----
    max_length=1024,  # cap on prompt + completion

    # ---- T4 16 GB hardening ----
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch = 8
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True, fp16=False,  # bf16 matmuls; no fp16 GradScaler pitfalls
    optim="paged_adamw_8bit",  # paged 8-bit optimizer => tiny state footprint

    # ---- Optimization schedule ----
    learning_rate=5e-6,  # DPO wants a SMALL LR; large LRs collapse the policy
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

# ref_model=None + peft_config => TRL builds pi_ref by DISABLING the LoRA adapters.
# ONE set of 4-bit base weights is BOTH policy and reference. This is why DPO fits a T4
# where PPO (4 separate models) does not. TRL also runs prepare_model_for_kbit_training here.
dpo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_ds,
    processing_class=tokenizer,  
    peft_config=peft_config,
)

### Step 4 — Train

Watch **`rewards/margins`** and **`rewards/accuracies`** climb — that is the policy learning to separate preferred from rejected completions.

In [ ]:
dpo_trainer.train()

# Save the aligned LoRA adapter (a few MB, not GB).
dpo_trainer.save_model("./dpo_aligned_adapter")
tokenizer.save_pretrained("./dpo_aligned_adapter")

## Export — Download the Aligned Adapter (Optional)

In [ ]:
import shutil, os

folder_to_zip = "./dpo_aligned_adapter"
output_filename = "dpo_aligned_adapter.zip"

shutil.make_archive(output_filename.replace(".zip", ""), "zip", folder_to_zip)
if os.path.exists(output_filename):
    print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")
else:
    print("Zip not found — run training + save first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did training + zipping finish?")

---

## Model Usage — Evaluate the DPO-Aligned Policy

Reload the **Instruct base + DPO adapter** and generate with the **same chat template** used in training. (Feeding an Instruct model raw prompts produces junk regardless of training quality — format must match.)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./dpo_aligned_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(base, adapter_path)  # attach the DPO adapter
model.eval()

In [ ]:
# Same chat template the DPO loop used (single user turn).
def generate_response(user_prompt, max_new_tokens=256, temperature=0.7):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Slice off the prompt tokens; decode only the newly generated completion.
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "Explain the difference between DPO and RLHF-PPO to a new ML engineer.",
    "My friend has been feeling really down lately. How can I support them?",
]

print("--- DPO-Aligned Responses ---")
for i, p in enumerate(test_prompts, 1):
    print(f"\n[Prompt {i}]: {p}")
    print(f"[Response]: {generate_response(p).strip()}")
    print("-" * 60)